In [1]:
import torch as th
import sys
import os
import matplotlib.pyplot as plt
from lpe.lpe.method_utils import *
from lpe.lpe.utils import Transformer
from lpe.lpe.utils import datasets as lpe_datasets
import numpy as np

from functools import partial

import jax.numpy as jnp
import jax 

from torch2jax import t2j

/home/hice1/jskifstad3/.conda/envs/mpx_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
## CONFIG

model_name = "gelu-1l"
device = th.device("cuda" if th.cuda.is_available() else "cpu")
model = Transformer.from_pretrained(model_name).to(device)
# initializing input distributions

dist_name = "camel"

# gt_freqs = load_ground_truth(model_name, [dist_name], device=device)[dist_name] # ground truth tensor
# gt_probs = gt_freqs / gt_freqs.sum()

lpe_datasets.USE_CACHE = True

: 

In [3]:
## UTILS

def gpt_obj(model,W_U_jn,N,W,target,x, u, t):
    # print(f"x: {x}")
    xdim = model.embed.d_model
    # print(f"xdim: {xdim}")

    # stage_cost = u.T @ W[xdim:,xdim:] @ u # for matrix W just penalizing scaled norm of u 
    stage_cost = u.T @ W[xdim:,xdim:] @ u # just penalizing scaled norm of u

    logits = x.T @ W_U_jn
    term_cost = jnp.abs(logits.argmax(-1) - logits[...,target])**2 + stage_cost

    return jnp.where(t == N, 0.5 * term_cost, 0.5 * stage_cost)
    # return stage_cost + term_cost

def gpt_hessian(model,W_U_jn,N,W,target,x, u, t):
    
    def residual(x,u):
        x_res = jnp.zeros_like(x).T
        # if (t == N):
            # logits = model.unembed(x).squeeze(1)
            # x_res = jnp.abs(logits.argmax(-1) - logits[...,target])**2
        logits = x.T @ W_U_jn
        term_res = jnp.abs(logits.argmax(-1) - logits[...,target])**2
        u_res = u.T
        return jnp.concatenate([jnp.where(t == N, term_res, x_res),u_res])
    
    J_x = jax.jacobian(residual,0)
    J_u = jax.jacobian(residual,1)
    return J_x(x,u).T@W@J_x(x,u), J_u(x,u).T@W@J_u(x,u), J_x(x,u).T@W@J_u(x,u)


def gpt_dynamics(model_blocks, x, u, t, parameter=None):
    print(f"in gpt dynamics: {t}")
    f_t = model_blocks[t]
    x_next = f_t(x) + u
    return x_next

In [7]:
## CONFIG

W_U_jn = jnp.asarray(model.unembed.W_U[None].detach())
tfs_torch = model.blocks
tfs_list = [t2j(block) for block in tfs_torch]
tfs_jax = jnp.asarray(tfs_list)


input = th.tensor([32990])

# Time and stage parameters -- do we care about anything other than N?
dt = 0.02  # Time step in seconds
N = len(model.blocks)    # Number of stages
mpc_frequency = 50  # Frequency of MPC updates in Hz

# Initial initial position (first embedding)
onehot = th.nn.functional.one_hot(input, num_classes=model.embed.d_vocab).float().to(device)
onehot.requires_grad_(True)
x = onehot @ model.embed.W_E
x = x + model.pos_embed(input)
# print(f"x shape: {x.shape}")

# p_numpy = x.detach().cpu().numpy() 
# p0 = jnp.array(p_numpy)
p0 = jnp.asarray(x.detach())

# Determine number of joints and contacts from the lists

n =  model.embed.d_model # Number of hidden dimensions
# print(f"NEW N: {n}")
m = n # Number of controls -- to be reduced?
grf_as_state = False
# Reference torques and controls (using n_joints)
u_ref = jnp.zeros(m)  # Reference controls (concatenated torques)

# Cost matrices (diagonal matrices created using jnp.diag)
Qp = jnp.diag(jnp.ones(n))  # Cost matrix for position
# Qp = jnp.ones(n)  # Cost matrix for position
# Qu = jnp.ones(m)  # Cost matrix for control
Qu = jnp.diag(jnp.ones(m))  # Cost matrix for control

print(Qp.size)
print(Qu.size)

# jnp.expand_dims(Qp,0)/
# jnp.expand_dims(Qu,0)
W = jax.scipy.linalg.block_diag(Qp, Qu)
# W = jnp.concatenate([Qp,Qu])

use_terrain_estimation = False  # Flag to use terrain estimation

cost = partial(gpt_obj, model, W_U_jn)
hessian_approx = partial(gpt_hessian, model, W_U_jn)
dynamics = partial(gpt_dynamics, tfs_jax)

target = 1537




TypeError: Value '<function t2j_module.<locals>.f at 0x7fffceeadd00>' with dtype object is not a valid JAX array type. Only arrays of numeric types are supported by JAX.

In [ ]:
import mpx.primal_dual_ilqr.primal_dual_ilqr.optimizers as optimizers
from timeit import default_timer as timer

def runOffline(x0, target):
    """
    Runs one MPC update using the current state, input, and foot positions.

    Args:
        x0: Current system state vector.

    Returns:
        A tuple (X, U, V) representing the computed state trajectory, control sequence,
        and auxiliary variable trajectory.
    """
    #compensate for the time delay
    #get forward kinematics for foot position
    print("starting")
    costy = partial(cost, N)
    work = partial(optimizers.mpc, costy, dynamics, hessian_approx, False) # set limited memory to false
    _solve = jax.jit(work)
    

    # self.X0 = self.X0.at[:,:13+self.config.n_joints].set(reference[:,:13+self.config.n_joints])

    _cost = partial(cost,N,W,target)
    _dynamics = dynamics
    model_evaluator = partial(optimizers.model_evaluator_helper, _cost, _dynamics,x0)
    jitted_model_evaluator = jax.jit(model_evaluator)

    _exit = False
    max_iter = 100
    last_cost = 1e10
    i = 0
    # output = []
    # output.append((self.X0))

    Xf = jnp.zeros([N+1, n])
    Uf = jnp.zeros([N, m])
    Vf = jnp.zeros([N+1, n]) #idk what this is
    print("starting the loop")
    while not _exit:
        start = timer()

        X, U, V = _solve(
            target,
            None, # parameter
            W,
            x0,
            Xf,
            Uf,
            Vf
            )
        print(f"post solve iteration: {i}")

        X.block_until_ready()

        Xf = X
        Uf = U
        Vf = V

        # output.append((self.X0))

        g, c = jitted_model_evaluator(X,U)

        stop = timer()

        l2_cost = np.sum(g*g)

        if i == 0:
            print("{:<10} {:<20} {:<20} {:<20}".format("Iter", "Cost", "Constraint", "Time Elapsed"))
        print("{:<10d} {:<20.5f} {:<20.5f} {:<20.5f}".format(i, l2_cost, np.sum(c*c), stop-start))
        i += 1

        if i > max_iter:
            print("exit because of max iter")
            _exit = True
        if last_cost - l2_cost < 1e-3 and np.sum(c*c) < 1e-5:
            print("exit because converged")
            _exit = True
        last_cost = l2_cost

    return Xf,Uf


# x0 = onehot @ config.model.embed.W_E
# x0 = x0 + config.model.pos_embed(config.input)



In [ ]:
X,U = runOffline(p0, target)

print(f"X: {X}")
print(f"U: {U}")

logits = model.unembed(X[-1, :]).squeeze(1)
print(f"final token: {logits.argmax(-1)}")

starting
starting the loop
in gpt dynamics: Traced<int32[]>with<BatchTrace> with
  val = Traced<int32[1]>with<DynamicJaxprTrace>
  batch_dim = 0


TracerIntegerConversionError: The __index__() method was called on traced array with shape int32[]
This BatchTracer with object id 140721988911472 was created on line:
  /home/hice1/jskifstad3/TrustworthyRoboticsLab/mpx/mpx/primal_dual_ilqr/primal_dual_ilqr/optimizers.py:600:30 (model_evaluator_helper)
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerIntegerConversionError